# RQ₃ — Holon-Based vs Naive GraphRAG: Reproducible Analysis

**Project 3 (Student C), BIL 930 Final.** Reproduces every table, statistic and figure of the §3.3 controlled experiment from the committed result CSVs. Deterministic on the default mock provider.

Regenerate the CSVs from scratch (offline):

```bash
python src/run_experiment.py
```

For real-LLM results set `P3_LLM_PROVIDER=openai|anthropic|ollama` (+ key) and re-run.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats as sps
from statsmodels.stats.contingency_tables import mcnemar

def find_p3():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if p.name == 'project_3_llm' and (p/'src'/'run_experiment.py').exists():
            return p
        if (p/'project_3_llm'/'src'/'run_experiment.py').exists():
            return p/'project_3_llm'
    raise RuntimeError('project_3_llm not found; run from the repo')

P3 = find_p3(); RES = P3/'results'
conf = pd.read_csv(RES/'accuracy'/'conformance.csv')
hall = pd.read_csv(RES/'hallucination'/'hallucination.csv')
eff  = pd.read_csv(RES/'performance'/'efficiency.csv')
print('questions:', len(conf)); conf.head()

## Deney 1 — SHACL Conformance (McNemar's paired test)

In [ ]:
def mc(a, b):
    n00=int(((a)&(b)).sum()); n01=int((a&~b).sum()); n10=int((~a&b).sum()); n11=int((~a&~b).sum())
    return [[n00,n01],[n10,n11]], mcnemar([[n00,n01],[n10,n11]], exact=True)

for label,(ca,cb) in {'first-pass':('fp_A','fp_B'),'converged':('conv_A','conv_B')}.items():
    a=conf[ca].astype(bool); b=conf[cb].astype(bool)
    tbl,res=mc(a,b)
    print(f'{label:11s}: A={a.mean():.0%} B={b.mean():.0%}  discordant A>B={tbl[0][1]} B>A={tbl[1][0]}  p={res.pvalue:.3e}')

## Deney 2 — Hallucination (assumption check → Wilcoxon, Cohen's d, bootstrap CI)

In [ ]:
hrA=hall['hr_A'].values; hrB=hall['hr_B'].values; diff=hrB-hrA
sh_p=sps.shapiro(diff)[1]
print(f'Shapiro p={sh_p:.2e} -> {"normal" if sh_p>0.05 else "non-normal"}')
if sh_p>0.05:
    t,p=sps.ttest_rel(hrB,hrA); print(f'paired t-test t={t:.3f} p={p:.3e}')
else:
    w,p=sps.wilcoxon(hrB,hrA); print(f'Wilcoxon W={w:.1f} p={p:.3e}')
d=diff.mean()/diff.std(ddof=0); print(f"Cohen's d={d:.2f}")
rng=np.random.default_rng(42)
boot=[rng.choice(hrB,len(hrB),replace=True).mean() for _ in range(2000)]
print(f'HR  A={hrA.mean():.3f}  B={hrB.mean():.3f}  95% CI(B)=[{np.percentile(boot,2.5):.3f},{np.percentile(boot,97.5):.3f}]')
print('Type A/B/C totals (B):', int(hall['B_typeA'].sum()), int(hall['B_typeB'].sum()), int(hall['B_typeC'].sum()))

## Deney 3 — Coverage & Token Efficiency

In [ ]:
print(f"Coverage   A={hall['covA'].mean():.2f}  B={hall['covB'].mean():.2f}")
print(f"Precision@3 A={hall['p3A'].mean():.2f}  B={hall['p3B'].mean():.2f}")
wA,wB=eff['tokEff_A'].values, eff['tokEff_B'].values
w,p=sps.wilcoxon(wA,wB)
print(f'Token efficiency A={wA.mean():.4f}  B={wB.mean():.4f}  Wilcoxon p={p:.3e}')
print(f"Mean context tokens A={eff['ctxTok_A'].mean():.0f}  B={eff['ctxTok_B'].mean():.0f}")

## Figures

In [ ]:
from IPython.display import Image, display
for fig in ['rq3_results.png','scalability.png']:
    p=RES/'figures'/fig
    if p.exists(): display(Image(filename=str(p)))

### Reading the result

- **First-pass conformance** is where SHACL-guided generation (A) dominates the naive baseline (B); McNemar rejects H₀.
- **Converged conformance** equalises once both systems get the k=3 SHACL-repair budget — an honest finding: the repair loop lets even the baseline reach structural validity. The lasting advantage of holon-based generation is **faithfulness**.
- **Hallucination** is eliminated by the holon boundary (A=0) vs the naive over-fetch (B), supporting H₃.
- **Token efficiency / context size**: the holon window keeps A's context small and constant as the corpus grows (see `scalability.png`).

See `../README.md` Threats to Validity for the mock-vs-real-LLM caveat.